# KNN Coffee Bean Predictor
Predicts a coffee bean (`name`) from roast level, aroma, structure, body, flavour, and aftertaste.

In [1]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import pandas as pd
import numpy as np
import re
import math
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [39]:
df = pd.read_csv('coffee_data.csv')
df['agtron_whole'] = df['agtron'].str.extract(r'^(\d+)').astype(float)
print(f'Shape: {df.shape}')
print(f'Roast levels: {df["roast_level"].unique()}')
print('Missing values:')
print(df.isnull().sum())
df.head()

Shape: (9000, 14)
Roast levels: ['Medium-Dark' 'Medium' 'Medium-Light' 'Light' 'Dark' 'Very Dark' nan]
Missing values:
name                   0
roaster                0
score                  4
roast_level          405
aroma                126
structure           5085
body                  68
flavour               72
aftertaste           847
coffee_origin        484
roaster_location       4
agtron                 0
cost                2012
agtron_whole         311
dtype: int64


,name,roaster,score,roast_level,aroma,structure,body,flavour,aftertaste,coffee_origin,roaster_location,agtron,cost,agtron_whole
0,Ka’u Classic Dark Roast,Rusty's Hawaiian,95.0,Medium-Dark,9.0,9.0,9.0,9.0,9.0,"Ka‘ū growing district, Hawai'i Island, Hawai'i","Pahala, Hawai'i Island, Hawai'i",44/64,$22.50/7 ounces,44.0
1,Kenya Peaberry Hera,Simon Hsieh Aroma Roast Coffees,95.0,Medium-Dark,9.0,9.0,9.0,9.0,9.0,"Nyeri growing region, south-central Kenya","Taiyuan, Taiwan",44/62,"NT $1,100/227 grams",44.0
2,Ethiopia Washed Limu,Kakalove Cafe,94.0,Medium-Dark,9.0,9.0,9.0,9.0,8.0,"Guji Zone, Oromia Region, southern Ethiopia","Chia-Yi, Taiwan",47/64,NT $290/8 ounces,47.0
3,Strawberry Shadow,modcup,94.0,Medium-Dark,9.0,8.0,9.0,9.0,9.0,"Huila and Quindío Departments, Colombia","Jersey City, New Jersey",46/63,$30.00/250 grams,46.0
4,Ethiopia Uraga Perfume Rose Anaerobic Washed,Linsun Coffee,93.0,Medium-Dark,9.0,8.0,9.0,9.0,8.0,"Guji Zone, Oromia Region, south-central Ethiopia","Tamsui, Taiwan",45/71,NT $800/200 grams,45.0


In [40]:
PROCESS_KEYWORDS = [
    'Carbonic Maceration', 'Anaerobic', 'Semi-Washed', 'Pulped Natural',
    'Wet-Hulled', 'Natural', 'Washed', 'Honey', 'Experimental',
]

VARIETY_KEYWORDS = [
    'Pink Bourbon', 'Yellow Bourbon', 'Red Bourbon',
    'SL28', 'SL34', 'Geisha', 'Gesha', 'Bourbon', 'Typica',
    'Pacamara', 'Caturra', 'Catuai', 'Sidra', 'Maragogipe',
    'Wush Wush', 'Mocca', 'Laurina', 'Maracaturra',
    'Mundo Novo', 'Peaberry', 'Marsellesa', 'Heirloom', 'Mejorado',
]

COUNTRY_KEYWORDS = [
    'Ethiopia', 'Kenya', 'Colombia', 'Panama', 'Guatemala', 'Costa Rica',
    'Honduras', 'El Salvador', 'Nicaragua', 'Peru', 'Brazil', 'Yemen',
    'Jamaica', "Hawai'i", 'Hawaii', 'Indonesia', 'Papua New Guinea',
    'Rwanda', 'Burundi', 'Uganda', 'Tanzania', 'Malawi', 'Zambia',
    'Ecuador', 'Bolivia', 'Mexico', 'China', 'Taiwan', 'India',
    'Myanmar', 'Thailand', 'Vietnam',
]

def extract_process(name):
    if pd.isna(name):
        return 'Unknown'
    for kw in PROCESS_KEYWORDS:
        if re.search(rf'\b{re.escape(kw)}\b', str(name), re.IGNORECASE):
            return kw
    return 'Unknown'

def extract_variety(name):
    if pd.isna(name):
        return 'Unknown'
    for kw in VARIETY_KEYWORDS:
        if re.search(rf'\b{re.escape(kw)}\b', str(name), re.IGNORECASE):
            return kw
    return 'Unknown'

def extract_country(origin):
    if pd.isna(origin):
        return 'Unknown'
    s = str(origin).lower()
    for country in COUNTRY_KEYWORDS:
        if country.lower() in s:
            return country
    return 'Unknown'

def extract_region(origin):
    if pd.isna(origin):
        return 'Unknown'
    parts = str(origin).split(',')
    region = parts[0].strip()
    return region if len(region) > 3 else 'Unknown'

df['process_method'] = df['name'].apply(extract_process)
df['variety']        = df['name'].apply(extract_variety)
df['coffee_country'] = df['coffee_origin'].apply(extract_country)
df['coffee_region']  = df['coffee_origin'].apply(extract_region)

print('process_method:', df['process_method'].value_counts().head(6).to_dict())
print('variety:',        df['variety'].value_counts().head(6).to_dict())
print('coffee_country:', df['coffee_country'].value_counts().head(6).to_dict())

process_method: {'Unknown': 7220, 'Natural': 928, 'Washed': 396, 'Honey': 248, 'Anaerobic': 174, 'Carbonic Maceration': 13}
variety: {'Unknown': 7749, 'Geisha': 375, 'Peaberry': 185, 'Gesha': 160, 'Pacamara': 105, 'Bourbon': 87}
coffee_country: {'Ethiopia': 2051, 'Unknown': 1525, 'Colombia': 1021, 'Kenya': 723, 'Guatemala': 579, 'Indonesia': 413}


In [41]:
NUMERIC_FEATURES = ['agtron_whole', 'aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score']
CAT_FEATURES     = ['coffee_country', 'coffee_region', 'process_method', 'variety']
FEATURES         = ['roast_level'] + NUMERIC_FEATURES + CAT_FEATURES

roast_order = [['Light', 'Medium-Light', 'Medium', 'Medium-Dark', 'Dark']]

preprocessor = ColumnTransformer(transformers=[
    ('roast', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(categories=roast_order, handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scale',  StandardScaler()),
    ]), ['roast_level']),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('categorical', Pipeline([
        ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=False)),
    ]), CAT_FEATURES),
])

print('Preprocessor defined.')

Preprocessor defined.


In [42]:
# Fit preprocessor on full dataset
X_raw = preprocessor.fit_transform(df[FEATURES])

# Build weight vector matching ColumnTransformer output order:
#   [roast(1)] + [numeric(7)] + [OHE columns per categorical feature]
NUMERIC_WEIGHTS = [0.8, 1.5, 1.2, 1.0, 2.0, 2.0, 1.0]  # agtron, aroma, structure, body, flavour, aftertaste, score
CAT_WEIGHTS     = [1.0, 0.8, 1.5, 1.5]                   # country, region, process, variety

# Use the ColumnTransformer's full output names to count OHE columns per feature.
# Names are prefixed 'categorical__' and may use either the column name or 'x{i}' format.
ct_names  = preprocessor.get_feature_names_out()
cat_names = [c for c in ct_names if c.startswith('categorical__')]

n_cols_per_cat = []
for i, feat in enumerate(CAT_FEATURES):
    n = sum(
        1 for c in cat_names
        if c.startswith(f'categorical__{feat}_') or c.startswith(f'categorical__x{i}_')
    )
    n_cols_per_cat.append(n)

ohe_weights = []
for n, w in zip(n_cols_per_cat, CAT_WEIGHTS):
    ohe_weights.extend([w] * n)

weight_vector = np.array([0.8] + NUMERIC_WEIGHTS + ohe_weights)

# Weighted feature matrix stored for similarity queries
X = X_raw * weight_vector

print(f'Feature matrix  : {X.shape}')
print(f'Weight vector   : {weight_vector.shape}')
print(f'OHE cols/feature: {dict(zip(CAT_FEATURES, n_cols_per_cat))}')

Feature matrix  : (9000, 222)
Weight vector   : (222,)
OHE cols/feature: {'coffee_country': 30, 'coffee_region': 156, 'process_method': 8, 'variety': 20}


In [43]:
knn = NearestNeighbors(metric='cosine')
knn.fit(X)
print('NearestNeighbors fitted on weighted feature matrix.')

NearestNeighbors fitted on weighted feature matrix.


In [44]:
def recommend_coffee(
        roast_level, aroma, structure, body, flavour, aftertaste,
        score=95, agtron_whole=None,
        coffee_country=None, coffee_region=None,
        process_method=None, variety=None,
        top_n=5):
    user_input = pd.DataFrame([{
        'roast_level':    roast_level,
        'agtron_whole':   agtron_whole,
        'aroma':          aroma,
        'structure':      structure,
        'body':           body,
        'flavour':        flavour,
        'aftertaste':     aftertaste,
        'score':          score,
        'coffee_country': coffee_country  or 'Unknown',
        'coffee_region':  coffee_region   or 'Unknown',
        'process_method': process_method  or 'Unknown',
        'variety':        variety          or 'Unknown',
    }])

    query               = preprocessor.transform(user_input) * weight_vector
    distances, indices  = knn.kneighbors(query, n_neighbors=top_n)
    similarities        = 1 - distances[0]

    names = df['name'].values
    print(f'Top {top_n} recommended coffees:')
    for rank, (idx, sim) in enumerate(zip(indices[0], similarities), 1):
        print(f'  {rank}. {names[idx]:<55s} {sim:.2f}')

print('recommend_coffee() ready.')

recommend_coffee() ready.


In [45]:
# Held-out evaluation — preprocessor fitted on full data, query test against train
df_tr, df_te = train_test_split(df, test_size=0.2, random_state=42)

X_tr     = preprocessor.transform(df_tr[FEATURES]) * weight_vector
X_te     = preprocessor.transform(df_te[FEATURES]) * weight_vector
names_tr = df_tr['name'].values
names_te = df_te['name'].values

top5 = top10 = 0
mrr_total = ndcg_total = 0.0

for i in range(len(X_te)):
    query     = X_te[i:i+1]
    true_name = names_te[i]
    sims      = cosine_similarity(query, X_tr)[0]
    ranked    = names_tr[np.argsort(sims)[::-1]]

    if true_name in ranked:
        rank = int(np.where(ranked == true_name)[0][0]) + 1
        if rank <= 5:  top5  += 1
        if rank <= 10: top10 += 1
        mrr_total  += 1.0 / rank
        if rank <= 10:
            ndcg_total += 1.0 / math.log2(rank + 1)

n = len(names_te)
print(f'Top-5  hit rate : {top5  / n:.4f}')
print(f'Top-10 hit rate : {top10 / n:.4f}')
print(f'MRR             : {mrr_total / n:.4f}')
print(f'NDCG@10         : {ndcg_total / n:.4f}')

Top-5  hit rate : 0.0306
Top-10 hit rate : 0.0422
MRR             : 0.0217
NDCG@10         : 0.0251


In [46]:
recommend_coffee('Light', aroma=9, structure=9, body=9, flavour=10, aftertaste=9,
                 process_method='Natural', variety='Geisha')

Top 5 recommended coffees:
  1. Ka‘ū Geisha Champagne Natural                           0.94
  2. Panama Don Pachi Natural Geisha                         0.93
  3. Panama Lino Esmeralda Geisha Natural                    0.93
  4. Panama Finca Kalithea Natural Geisha                    0.91
  5. Esmeralda Estate Porton Geisha Natural                  0.91


## V5 — Weighted Cosine Retrieval Recommender (KNN)

V5 reframes the model as a content-based coffee recommender. Instead of predicting one exact coffee bean, it builds weighted feature vectors from tasting scores, roast data, score, origin, process and variety, then retrieves the most similar coffees using `NearestNeighbors(metric='cosine')`. This better matches the real problem because many coffees share near-identical tasting profiles, making exact classification unreliable.

### Features
| Feature | Weight | Notes |
|---|---|---|
| flavour | 2.0 | Primary similarity signal |
| aftertaste | 2.0 | Primary similarity signal |
| aroma | 1.5 | |
| process_method | 1.5 | Extracted from coffee name |
| variety | 1.5 | Extracted from coffee name |
| structure | 1.2 | |
| body | 1.0 | |
| score | 1.0 | |
| coffee_country | 1.0 | Extracted from coffee_origin |
| agtron_whole | 0.8 | |
| roast_level | 0.8 | |
| coffee_region | 0.8 | Extracted from coffee_origin |

### Changes from V4
- Dropped `roaster` and `roaster_location` (high-cardinality noise — confirmed damaging in V3)
- Added `process_method`, `variety`, `coffee_country`, `coffee_region` extracted from text
- Feature weights applied after preprocessing so flavour/aftertaste/aroma drive similarity
- OHE uses `min_frequency=10` to avoid rare one-off categories inflating the vector space
- Missing categoricals filled with `'Unknown'` (not most-frequent imputation)
- Uses `NearestNeighbors(metric='cosine')` not `KNeighborsClassifier`
- Evaluation uses Top-5/10 hit rate, MRR, NDCG@10 — not exact-match accuracy